# 21e — Series 21: μ REINFORCE **reward** redesign (responsibility vs per-run likelihood vs count-corrected credit)

**Series 21 (supervised). ONE change vs 21a: the μ REINFORCE *reward* only.** Everything else frozen —
17g model (`SIGMA_REF=10`, z-form γ-score `H_REF=1`, `LR_MU=15` linear decay, `LR_GAMMA=0.5` annealed,
`CLIP=10`, `LAMBDA_MEAN=0`, `H_S_MIN=0.05`, `SEED=42`, init `0.5×truth`), series-20 budget
`N_RUNS=100`, `N_ITER=30`, real-data targets (15-series load & filter).

## Why
21a/21c showed μ is a *dead / biased* channel: μ ends at 0.41–0.55× truth in all 14 experiments; the
init-sensitivity test gave μ ≈ init (0.5→0.52, 1.5→1.50); and the μ update is a REINFORCE covariance

    grad_mu = −(B − mean B)·score ,   score = (n−μ)/SIGMA_REF² ,   B_j = mean_i w_ij

where `w_ij = W_ij / Σ_j' W_ij'` is the **row-normalised** responsibility (a probability distribution over
the 100 sim runs). When the KDE kernel is flat (`h` large) `w_ij → 1/M` for every draw, hence
`B − mean B → 0` and the gradient vanishes *identically*. Measured: `sd(B) ≈ 0.002–0.01`, effective #draws
carrying the mass 78–97/100, and `corr(B_j, n_j)` flips sign across experiments (−0.3…+0.34).

## Hypothesis (Anuar)
The reward should be the **likelihood of the real data under that specific run's Gaussian**, not the
normalised responsibility:

    R_j = ∏_i N( x_i | FWHM_j, σ_fit_j, h )   ⇔   log R_j = Σ_i log W_ij .

This keeps the reward **magnitude** (no dilution by the other 99 draws), so the reward↔count coupling
should survive even when the kernel is flat.

## Arms  (`grad_mu = −(r − mean r)·score`)
| arm | reward r_j | note |
|-----|-----------|------|
| `responsibility` | `mean_i w_ij` (B) | current 21a — **control, must reproduce 21c** |
| `loglik` | `mean_i log W_ij` | the proposal, raw |
| `corr` | `mean_i w_ij · ∂logW_ij/∂n_j` | exact per-draw credit (count-sensitivity kept) |
| `loglik_z` | z-score of (loglik) per batch | scale-free (raw loglik is ~10³× B and can saturate CLIP) |

## Prediction (falsifiable)
(a) `responsibility` reproduces 21c; (b) raw `loglik` has |grad| ≫ CLIP=10 → saturates; (c) `loglik_z` has
a **stable-sign** reward↔count coupling (`corr(logW, n) > 0`) ⇒ μ actually moves, drifting **up** from
0.5×; (d) open question: does it land nearer truth or overshoot? γ has a pathwise (reparameterised)
gradient, so γ results should be unchanged vs 21a — a good internal control.


## Panel
1. **FIG 1** μ/mu_true vs transmission, one line per arm (per power).
2. **FIG 2** μ trajectory vs step per experiment (arms overlaid) — does μ move at all?
3. **FIG 3** coverage |dmu|/sigma_mu vs transmission per arm (+ 1σ/2σ bands).
4. **FIG 4** gamma/gamma_true vs transmission per arm (expect ≈unchanged — pathwise control).
5. **FIG 5** raw pre-clip |grad_mu| and clipped fraction per arm.
6. **FIG 6** reward↔count coupling corr(r_j, n_j) vs transmission per arm.
7. **TABLE** per-arm μ/γ rel-RMSE, bias, coverage.

Convention: figures render **inline only** (no savefig); the notebook is executed **in place**.


In [ ]:
import math, time, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p)
        REPO_ROOT = _p
        break
os.chdir(REPO_ROOT)

from src.fitting import nll, fwhm_from_theta, fit_profile
from src.samplers import draw_fixed_noise, build_photons
from src.implicit import compute_fwhm_and_dgamma

print('Imports OK')

In [ ]:
# ============================================================
# EXPERIMENTS — true values from Gregor's fits (identical to 17f/16-series) + real data file
# ============================================================
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393, sigma_prop=2.576, lam=2.232, gamma_true=8.5, n_target=61, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans05.txt'),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372, sigma_prop=3.445, lam=2.122, gamma_true=8.5, n_target=358, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans10.txt'),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316, sigma_prop=4.141, lam=2.286, gamma_true=8.5, n_target=1138, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans20.txt'),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405, sigma_prop=7.198, lam=2.351, gamma_true=8.5, n_target=2428, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans40.txt'),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374, sigma_prop=9.851, lam=2.593, gamma_true=8.5, n_target=2424, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans60.txt'),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365, sigma_prop=12.627, lam=2.758, gamma_true=8.5, n_target=2487, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans80.txt'),
    dict(name='1nW Trans100',  power='1nW', mu_true=70.817, sigma_prop=17.221, lam=2.636, gamma_true=8.5, n_target=2455, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans100.txt'),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204, sigma_prop=3.724, lam=2.186, gamma_true=14.1, n_target=252, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans05.txt'),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476, sigma_prop=5.639, lam=2.158, gamma_true=14.1, n_target=1572, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans10.txt'),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279, sigma_prop=8.319, lam=2.264, gamma_true=14.1, n_target=2171, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans20.txt'),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892, sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans40.txt'),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95, lam=2.741, gamma_true=14.1, n_target=2541, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans60.txt'),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans80.txt'),
    dict(name='3nW Trans100',  power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans100.txt'),
]

# ---- QUICK TEST: uncomment one of these slices ----
# EXPERIMENTS = EXPERIMENTS[:3]                                        # first 3
# EXPERIMENTS = [e for e in EXPERIMENTS if e['name'] in
#                ('1nW Trans80', '3nW Trans80', '3nW Trans100')]      # the hard ones

print(f'{len(EXPERIMENTS)} experiments configured')


In [ ]:
# ============================================================
# SERIES 21 — 21e CONFIG: one change vs 21a = the mu REINFORCE reward
# ============================================================
N_RUNS   = 100
N_ITER   = 30
LR_MU    = 15.0
LR_GAMMA = 0.5
SIGMA_REF = 10.0
GAMMA_SCALE = True
H_REF = 1.0
CLIP     = 10.0
SEED     = 42
LAMBDA_MEAN = 0.0
H_S_MIN = 0.05

# ---- THE 21e CHANGE: which reward weights the mu REINFORCE score ----
REWARD_MODES = ['responsibility', 'loglik', 'corr', 'loglik_z']

# ---- FISHER / CRB (15-series protocol) ----
M_FINAL = 500
FISHER_SEEDS = 1
FISHER_BASE_SEED = 7000

# ---- SMOKE: NB_SMOKE=1 -> fast subset (validates machinery + all figures) ----
SMOKE = os.environ.get('NB_SMOKE') == '1'
if SMOKE:
    N_RUNS, N_ITER, M_FINAL = 10, 5, 50
    EXPERIMENTS = EXPERIMENTS[:3]
    print('*** SMOKE RUN ***')

N_CPUS = os.cpu_count() or 4
N_EXP_PARALLEL = 1
N_WORKERS = 4
if N_EXP_PARALLEL * N_WORKERS > N_CPUS:
    N_WORKERS = max(1, N_CPUS // N_EXP_PARALLEL)

print(f'config: N_RUNS={N_RUNS}, N_ITER={N_ITER}, reward modes={REWARD_MODES}')
print(f'fisher: M_FINAL={M_FINAL}, FISHER_SEEDS={FISHER_SEEDS}')
print(f'parallelism: {N_EXP_PARALLEL} exp(s) x {N_WORKERS} workers on {N_CPUS} CPUs')


In [ ]:
# ============================================================
# helpers: KDE scores (unchanged from 21a) + THE 21e reward vectors
# ============================================================
def _fit_fn(ph):
    return fit_profile(ph, n_iters=80, model='lorentzian', uniform_bg=False)
def _fwhm_fn(th):
    return fwhm_from_theta(th, model='lorentzian')
def _nll_fn(th, ph):
    return nll(th, ph, model='lorentzian', uniform_bg=False)

def kde_scores(sim_f, sim_s, sim_n, sim_df, sim_ds, data_f, data_s, h_f, h_s, mu, sigma_prop):
    d_f = data_f[:, None] - sim_f[None, :]
    d_s = data_s[:, None] - sim_s[None, :]
    W = torch.exp(-0.5 * (d_f / h_f) ** 2 - 0.5 * (d_s / h_s) ** 2)
    w = W / W.sum(dim=1, keepdim=True).clamp_min(1e-12)
    score = (sim_n[None, :] - mu) / sigma_prop ** 2
    s_mu = (w * score).sum(dim=1)
    if GAMMA_SCALE:
        dlogG = ((d_f * sim_df[None, :]) / h_f + (d_s * sim_ds[None, :]) / h_s) / H_REF
    else:
        dlogG = (d_f * sim_df[None, :]) / h_f ** 2 + (d_s * sim_ds[None, :]) / h_s ** 2
    s_gamma = (w * dlogG).sum(dim=1)
    logp = torch.log((W.sum(dim=1) / len(sim_f)).clamp_min(1e-30))
    return s_mu, s_gamma, -logp.mean(), w, W

def reward_vector(mode, W, w, sim_s, sim_n, data_s, h_s):
    '''Per-sim-draw REINFORCE reward r_j. THE 21e CHANGE lives here.'''
    if mode == 'responsibility':                 # 21a control: row-normalised responsibility B_j
        return w.mean(dim=0)
    if mode == 'loglik':                         # proposal: per-run Gaussian log-likelihood of the data
        return torch.log(W.clamp_min(1e-30)).mean(dim=0)
    if mode == 'loglik_z':                       # proposal, scale-free (direction only)
        r = torch.log(W.clamp_min(1e-30)).mean(dim=0)
        return (r - r.mean()) / (r.std() + 1e-12)
    if mode == 'corr':                           # exact per-draw credit dF/dn_j (count-sensitivity kept)
        d_s = data_s[:, None] - sim_s[None, :]
        dsig_dn = -(sim_s[None, :]) / (2.0 * sim_n[None, :].clamp_min(1.0))   # sigma_fit ~ c/sqrt(n)
        dlogW_dn = (d_s / h_s ** 2) * dsig_dn
        return (w * dlogW_dn).mean(dim=0)
    raise ValueError(mode)

import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE
def _run_one(args):
    gamma_val, u, b = args
    return compute_fwhm_and_dgamma(gamma_val, u, b, _fit_fn, _fwhm_fn, _nll_fn, n_params=2)
def _init_worker():
    torch.set_num_threads(1)
def _parallel_map(pool, tasks):
    return list(pool.map(_run_one, tasks, chunksize=8))
print('helpers ready')


In [ ]:
# ============================================================
# One experiment under a chosen mu reward mode (otherwise identical to 21a)
# ============================================================
def load_target(exp):
    d = np.genfromtxt(exp['data_file'])
    fwhm_mhz = d[:, 0] * 1000.0; err_mhz = d[:, 1] * 1000.0
    ok = ~np.isnan(fwhm_mhz) & ~np.isnan(err_mhz) & (fwhm_mhz > 0)
    filt = ok & (err_mhz / fwhm_mhz < 10.0)
    tf = torch.tensor(fwhm_mhz[filt], dtype=torch.float32)
    ts = torch.tensor(err_mhz[filt], dtype=torch.float32)
    n = len(tf); sc = n ** (-1.0 / 6.0)
    hf = float(tf.std()) * sc
    hs = max(float(ts.std()) * sc, H_S_MIN)
    return tf, ts, hf, hs

def run_experiment(exp, reward_mode, seed_base=None):
    mu_true, sigma_prop = exp['mu_true'], exp['sigma_prop']
    lam, gamma_true = exp['lam'], exp['gamma_true']
    _seed = SEED if seed_base is None else seed_base
    mu_val, gamma_val = 0.5 * mu_true, 0.5 * gamma_true
    target_f, target_s, H_F, H_S = load_target(exp)
    n_target = len(target_f)

    with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'),
              initializer=_init_worker) as pool:
        history = []; t0 = time.time()
        for step in range(N_ITER):
            rng2 = np.random.default_rng(_seed + step)
            tasks, ns = [], []
            for _ in range(N_RUNS):
                u, b, n = draw_fixed_noise(mu_val, sigma_prop, lam, rng2)
                tasks.append((gamma_val, u.numpy(), b.numpy())); ns.append(n)
            res = _parallel_map(pool, tasks)
            ft = torch.tensor([r[0] for r in res], dtype=torch.float32)
            si = torch.tensor([r[1] for r in res], dtype=torch.float32)
            nt = torch.tensor(ns, dtype=torch.float32)
            dg_t = torch.tensor([r[2] for r in res], dtype=torch.float32)
            ds_t = torch.tensor([r[3] for r in res], dtype=torch.float32)

            s_mu, s_gamma, nll_val, w, W = kde_scores(
                ft, si, nt, dg_t, ds_t, target_f, target_s, H_F, H_S, mu_val, sigma_prop)

            # ---- mu: REINFORCE with the 21e reward ----
            r = reward_vector(reward_mode, W, w, si, nt, target_s, H_S)
            score = (nt - mu_val) / SIGMA_REF ** 2
            grad_mu_raw = float(-(r - r.mean()) @ score)
            grad_mu = float(max(min(grad_mu_raw, CLIP), -CLIP))
            clipped = abs(grad_mu_raw) > CLIP
            corr_rn = 0.0
            if float(r.std()) > 0 and float(nt.std()) > 0:
                corr_rn = float(((r - r.mean()) * (nt - nt.mean())).mean() / (r.std() * nt.std()))

            # ---- gamma: KDE channel (unchanged) ----
            grad_gamma = float(max(min(-s_gamma.mean(), CLIP), -CLIP))

            # ---- updates (17f, unchanged) ----
            lr_mu_decay = LR_MU * (1.0 - step / N_ITER)
            mu_val = max(1.0, min(200.0, mu_val - lr_mu_decay * grad_mu))
            gamma_val = max(0.1, min(100.0, gamma_val - LR_GAMMA * (1.0 - 0.5 * step / N_ITER) * grad_gamma))
            history.append(dict(step=step, mu=mu_val, gamma=gamma_val, nll=float(nll_val),
                                grad_mu=grad_mu, grad_mu_raw=grad_mu_raw, clipped=bool(clipped),
                                corr_rn=corr_rn, r_std=float(r.std()), mean_n=float(nt.mean())))
        # ---- Fisher / CRB at the fitted point (15-series protocol) ----
        J_list = []
        for seed in range(FISHER_SEEDS):
            rng = np.random.default_rng(FISHER_BASE_SEED + seed)
            tasks, ns = [], []
            for _ in range(M_FINAL):
                u, b_, n = draw_fixed_noise(mu_val, sigma_prop, lam, rng)
                tasks.append((gamma_val, u.numpy(), b_.numpy())); ns.append(n)
            res2 = _parallel_map(pool, tasks)
            ft = torch.tensor([x[0] for x in res2], dtype=torch.float32)
            si2 = torch.tensor([x[1] for x in res2], dtype=torch.float32)
            nn2 = torch.tensor(ns, dtype=torch.float32)
            dg2 = torch.tensor([x[2] for x in res2], dtype=torch.float32)
            ds2 = torch.tensor([x[3] for x in res2], dtype=torch.float32)
            s_mu2, s_g2, _, _, _ = kde_scores(ft, si2, nn2, dg2, ds2,
                                              target_f, target_s, H_F, H_S, mu_val, sigma_prop)
            s = torch.stack([s_mu2, s_g2], dim=1)
            J_list.append(s.T @ s / len(target_f))
        J = torch.stack(J_list).mean(dim=0)
        Jinv = torch.linalg.inv(J + 1e-8 * torch.eye(2))
        std_mu = math.sqrt(max(Jinv[0, 0].item(), 0.0)); std_g = math.sqrt(max(Jinv[1, 1].item(), 0.0))
        corr = Jinv[0, 1].item() / math.sqrt(max(Jinv[0, 0].item() * Jinv[1, 1].item(), 1e-30))

    return dict(exp=exp['name'], power=exp['power'], reward_mode=reward_mode, T=int(exp['name'].split('Trans')[1]),
                mu_true=mu_true, gamma_true=gamma_true, sigma_prop=sigma_prop,
                mu_final=history[-1]['mu'], gamma_final=history[-1]['gamma'], nll_final=history[-1]['nll'],
                std_mu=std_mu, std_gamma=std_g, corr=corr, history=history, n_target=n_target)

print('run_experiment ready (reward-mode aware)')


In [ ]:
# ============================================================
# RUN: every (reward mode, experiment) pair
# ============================================================
t_all = time.time()
results = []
for mode in REWARD_MODES:
    print(f'--- reward = {mode} ---')
    for exp in EXPERIMENTS:
        r = run_experiment(exp, mode)
        results.append(r)
        print(f"{exp['name']:>12}: mu {r['mu_true']:7.2f} -> {r['mu_final']:7.2f} (x{r['mu_final']/r['mu_true']:.2f})"
              f" | gamma {r['gamma_true']:5.1f} -> {r['gamma_final']:6.2f} | {r['history'][-1]['nll']:.2f}")
print(f'\nTotal: {(time.time()-t_all)/60:.1f} min')


In [ ]:
# ============================================================
# FIG 1 — mu/mu_true vs transmission, one line per reward arm
# ============================================================
by = {}
for r in results:
    by.setdefault(r['reward_mode'], []).append(r)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, p in zip(axes, ['1nW', '3nW']):
    for mode in REWARD_MODES:
        rs = sorted([r for r in by[mode] if r['power'] == p], key=lambda z: z['T'])
        ax.plot([r['T'] for r in rs], [r['mu_final'] / r['mu_true'] for r in rs], 'o-', label=mode)
    ax.axhline(1.0, color='k', ls='--', lw=1); ax.set_title(f'mu/mu_true — {p}')
    ax.set_xlabel('Transmission (%)'); ax.set_ylabel('mu_hat/mu_true'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('FIG 1 — mu accuracy vs transmission, per reward arm')
plt.tight_layout(); plt.show()

# ============================================================
# FIG 2 — mu trajectory vs step per experiment (arms overlaid)
# ============================================================
exps = [e['name'] for e in EXPERIMENTS]
ncol = 4; nrow = int(math.ceil(len(exps) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow))
axes = np.asarray(axes).reshape(nrow, ncol)
for k, nm in enumerate(exps):
    ax = axes[k // ncol][k % ncol]
    for mode in REWARD_MODES:
        rr = [r for r in by[mode] if r['exp'] == nm][0]
        ax.plot([h['step'] for h in rr['history']], [h['mu'] / rr['mu_true'] for h in rr['history']], '-', label=mode)
    ax.axhline(1.0, color='k', ls='--', lw=0.8); ax.set_title(nm, fontsize=9)
    ax.set_xlabel('step'); ax.set_ylabel('mu/mu_true'); ax.grid(alpha=0.3)
    if k == 0: ax.legend(fontsize=6)
for k in range(len(exps), nrow * ncol): axes[k // ncol][k % ncol].axis('off')
plt.suptitle('FIG 2 — mu trajectories per experiment (one line per arm)')
plt.tight_layout(); plt.show()

# ============================================================
# FIG 3 — coverage |dmu|/sigma_mu vs transmission per arm
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, p in zip(axes, ['1nW', '3nW']):
    for mode in REWARD_MODES:
        rs = sorted([r for r in by[mode] if r['power'] == p], key=lambda z: z['T'])
        ax.plot([r['T'] for r in rs],
                [abs(r['mu_final'] - r['mu_true']) / r['std_mu'] for r in rs], 'o-', label=mode)
    ax.axhline(1, color='k', ls=':', lw=1); ax.axhline(2, color='k', ls='--', lw=1)
    ax.set_title(f'|dmu|/sigma_mu — {p}'); ax.set_xlabel('Transmission (%)')
    ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('FIG 3 — mu coverage vs transmission (dotted=1σ, dashed=2σ)')
plt.tight_layout(); plt.show()

# ============================================================
# FIG 4 — gamma/gamma_true vs transmission per arm (pathwise control)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, p in zip(axes, ['1nW', '3nW']):
    for mode in REWARD_MODES:
        rs = sorted([r for r in by[mode] if r['power'] == p], key=lambda z: z['T'])
        ax.plot([r['T'] for r in rs], [r['gamma_final'] / r['gamma_true'] for r in rs], 'o-', label=mode)
    ax.axhline(1.0, color='k', ls='--', lw=1); ax.set_title(f'gamma/gamma_true — {p}')
    ax.set_xlabel('Transmission (%)'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('FIG 4 — gamma accuracy vs transmission (should be ~unchanged across arms)')
plt.tight_layout(); plt.show()

# ============================================================
# FIG 5 — raw pre-clip |grad_mu| and clipped fraction per arm
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for mode in REWARD_MODES:
    rs = sorted([r for r in by[mode] if r['power'] == '1nW'], key=lambda z: z['T'])
    axes[0].plot([r['T'] for r in rs], [np.median([abs(h['grad_mu_raw']) for h in r['history']]) for r in rs], 'o-', label=mode)
    axes[1].plot([r['T'] for r in rs], [np.mean([h['clipped'] for h in r['history']]) for r in rs], 'o-', label=mode)
axes[0].set_title('median |grad_mu_raw| (1nW)'); axes[0].set_yscale('log')
axes[1].set_title('fraction of steps clipped (1nW)'); axes[1].set_ylim(-0.02, 1.02)
for ax in axes:
    ax.set_xlabel('Transmission (%)'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('FIG 5 — gradient scale & CLIP saturation per arm')
plt.tight_layout(); plt.show()

# ============================================================
# FIG 6 — reward<->count coupling corr(r_j, n_j) vs transmission
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, p in zip(axes, ['1nW', '3nW']):
    for mode in REWARD_MODES:
        rs = sorted([r for r in by[mode] if r['power'] == p], key=lambda z: z['T'])
        ax.plot([r['T'] for r in rs], [np.mean([h['corr_rn'] for h in r['history']]) for r in rs], 'o-', label=mode)
    ax.axhline(0, color='k', lw=0.6); ax.set_title(f'corr(r_j, n_j) — {p}')
    ax.set_xlabel('Transmission (%)'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('FIG 6 — reward<->count coupling (the quantity the mu gradient lives on)')
plt.tight_layout(); plt.show()

# ============================================================
# FIG 7 — per-arm summary table (mu/gamma rel-RMSE, bias, coverage)
# ============================================================
print(f"{'arm':>15} | {'mu rel-RMSE':>11} {'mu rel-bias':>11} {'mu cov<2s':>9} | {'g rel-RMSE':>10} {'g rel-bias':>10}")
print('-' * 84)
for mode in REWARD_MODES:
    rs = by[mode]
    rmu = np.array([r['mu_final'] / r['mu_true'] - 1 for r in rs])
    rg = np.array([r['gamma_final'] / r['gamma_true'] - 1 for r in rs])
    cov = np.mean([abs(r['mu_final'] - r['mu_true']) / r['std_mu'] <= 2 for r in rs])
    print(f"{mode:>15} | {np.sqrt((rmu**2).mean())*100:11.1f} {rmu.mean()*100:11.1f} {cov*len(rs):6.0f}/{len(rs)} | "
          f"{np.sqrt((rg**2).mean())*100:10.1f} {rg.mean()*100:10.1f}")


## Verdict — to be filled from the executed outputs above

(placeholder: replaced after the run with the arm-by-arm read: does mu move, in which direction, where does
it land, does coverage survive, and does the pathwise-gamma control hold.)
